El presente objetivo del trabajo es crear una app de análisis de sentimientos dessde un texto ingresado por el usuario

Se utilizara Pipeline de transformers desde Hugging Face 

<u><h1>Primera Parte: Dos modelos con Pipeline</u></h1>

<h2>Modelo 1</h2>  

Nombre: nlptown/bert-base-multilingual-uncased-sentiment  

* Link al Models https://huggingface.co/nlptown/bert-base-multilingual-uncased-sentiment   

* Modelo que utiliza: Bert  

* Idiomas que soporta: Ingles, Neerlandés, Aleman, Francés, Español, e Italiano  

In [8]:
from transformers import pipeline

pipe1 = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment")

Loading weights: 100%|██████████| 201/201 [00:01<00:00, 111.80it/s]


In [80]:
resultado_pipeline_1 = pipe1("Me encantó el producto, es excelente.")

print(resultado_pipeline_1)

[{'label': '5 stars', 'score': 0.8544487357139587}]


<h2>Modelo 2</h2>
Nombre: 🚀 Multilingual Sentiment Classification Model (23 Languages)

* Link al Models: https://huggingface.co/tabularisai/multilingual-sentiment-analysis

* Modelo que utiliza: distilbert/distilbert-base-multilingual-cased

* Idiomas: que soporta: English plus Chinese (中文), Spanish (Español), Hindi (हिन्दी), Arabic (العربية), Bengali (বাংলা), Portuguese (Português), Russian (Русский), Japanese (日本語), German (Deutsch), Malay (Bahasa Melayu), Telugu (తెలుగు), Vietnamese (Tiếng Việt), Korean (한국어), French (Français), Turkish (Türkçe), Italian (Italiano), Polish (Polski), Ukrainian (Українська), Tagalog, Dutch (Nederlands), Swiss German (Schweizerdeutsch), and Swahili.


In [6]:
from transformers import pipeline

# Load the classification pipeline with the specified model
pipe2 = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 358.72it/s]


In [81]:
# Classify a new sentence
sentence = "Me encantó el producto, es excelente."
resultado_pipeline_2 = pipe2(sentence)

# Print the result
print(resultado_pipeline_2)

[{'label': 'Positive', 'score': 0.8007412552833557}]


Ambos modelos lograron clasificar correctamente la misma frase, la diferencia principal con el modelo 2 es que no cuenta con "stars" como puntaje, sino con categorias "Very Negative, Negative, Neutral, Positive, Very Positive".   
Debido a que ambas son 5 niveles de puntaje se puede establecer esta comparacion: 
1 star       ↔ Very Negative  
2 stars      ↔ Negative  
3 stars      ↔ Neutral  
4 stars      ↔ Positive  
5 stars      ↔ Very Positive  

<u><h1>Segunda Parte: Los modelos con Inference Client</u></h1>

<h2>Modelo 1</h2> 

In [82]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

resultado_inference_1 = client.text_classification(
    "Me gustas mucho, te amo",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)

KeyError: 'HF_TOKEN'

In [ ]:
import os

print(os.environ.get("HF_TOKEN"))

'''La posibilidad es crear la carpeta env con el token o usar get pass para poder pegarla y que no se vea'''

None


In [ ]:
from getpass import getpass
from huggingface_hub import InferenceClient

token = getpass("Pegá tu Hugging Face token: ")

client1 = InferenceClient(
    provider="hf-inference",
    api_key=token,
)

resultado_inference_1 = client.text_classification(
    "Me gustas mucho, te amo",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)

print(resultado_inference_1)
print(type(resultado_inference_1))
print(type(resultado_inference_1[0]))
print(type(resultado_inference_1[0].label))
print(type(resultado_inference_1[0].score))

[TextClassificationOutputElement(label='5 stars', score=0.571591854095459), TextClassificationOutputElement(label='4 stars', score=0.3533557653427124), TextClassificationOutputElement(label='3 stars', score=0.06350746005773544), TextClassificationOutputElement(label='2 stars', score=0.006665526889264584), TextClassificationOutputElement(label='1 star', score=0.004879408981651068)]
<class 'list'>
<class 'huggingface_hub.inference._generated.types.text_classification.TextClassificationOutputElement'>
<class 'str'>
<class 'float'>


<h2>Modelo 2</h2> 

In [84]:
import os
from huggingface_hub import InferenceClient

token = getpass("Pegá tu Hugging Face token: ")

client2 = InferenceClient(
    provider="hf-inference",
    api_key=token,
)

resultado_inference_2 = client.text_classification(
    "I like you. I love you",
    model="tabularisai/multilingual-sentiment-analysis",
)

print(resultado_inference_2)
print(type(resultado_inference_2))
print(type(resultado_inference_2[0]))
print(type(resultado_inference_2[0].label))
print(type(resultado_inference_2[0].score))

[TextClassificationOutputElement(label='Very Positive', score=0.6660197973251343), TextClassificationOutputElement(label='Positive', score=0.23012897372245789), TextClassificationOutputElement(label='Neutral', score=0.061766646802425385), TextClassificationOutputElement(label='Negative', score=0.022080767899751663), TextClassificationOutputElement(label='Very Negative', score=0.02000380866229534)]
<class 'list'>
<class 'huggingface_hub.inference._generated.types.text_classification.TextClassificationOutputElement'>
<class 'str'>
<class 'float'>


<u><h1>Tercera Parte: Frases de Prueba</u></h1>

<h2>Modelo 1 Pipeline vs Inference Client</h2> 

In [62]:
import time

In [63]:
frases = [
    "Me encanto el producto, es excelente",
    "No me gusto el producto, la calidad es muy mala",
    "Es genial, recomendados",
    "Es horrible, nunca volveria a comprarlo",
    ".",
    "Cumple con las 3 B, bueno, bonito, barato",
    "No se, era para un regalo",
    "Lo barato sale caro",
    "Ni ACME se atrevio a tanto"
]

In [72]:
#con pipeline-modelo 1
resultados = []

for frase in frases:
    inicio = time.perf_counter()

    resultado = pipe1(frase)

    fin = time.perf_counter()

    tiempo = fin - inicio

    resultados.append({
        "frase": frase,
        "resultado": resultado[0],
        "tiempo_segundos": tiempo
    })

resultados_pipeline_1 = resultados
resultados_pipeline_1

[{'frase': 'Me encanto el producto, es excelente',
  'resultado': {'label': '5 stars', 'score': 0.8410688638687134},
  'tiempo_segundos': 0.5392273000034038},
 {'frase': 'No me gusto el producto, la calidad es muy mala',
  'resultado': {'label': '1 star', 'score': 0.7105898261070251},
  'tiempo_segundos': 0.15196969998942222},
 {'frase': 'Es genial, recomendados',
  'resultado': {'label': '5 stars', 'score': 0.7857764959335327},
  'tiempo_segundos': 0.1403893000097014},
 {'frase': 'Es horrible, nunca volveria a comprarlo',
  'resultado': {'label': '1 star', 'score': 0.9611223936080933},
  'tiempo_segundos': 0.14374149999639485},
 {'frase': '.',
  'resultado': {'label': '3 stars', 'score': 0.2539339065551758},
  'tiempo_segundos': 0.057378700003027916},
 {'frase': 'Cumple con las 3 B, bueno, bonito, barato',
  'resultado': {'label': '4 stars', 'score': 0.4829288721084595},
  'tiempo_segundos': 0.09211509999295231},
 {'frase': 'No se, era para un regalo',
  'resultado': {'label': '1 star

In [73]:
#Log de error con inference Client-modelo 1

resultados = []

for frase in frases:
    inicio = time.perf_counter()

    resultado = client1(frase)

    fin = time.perf_counter()

    tiempo = fin - inicio

    resultados.append({
        "frase": frase,
        "resultado": resultado[0],
        "tiempo_segundos": tiempo
    })

resultados_inference_1= resultados
resultados_inference_1


TypeError: 'InferenceClient' object is not callable

In [74]:
#con inference Client-modelo 1
resultados = []

for frase in frases:
    inicio = time.perf_counter()

    resultado = client1.text_classification(
        frase,
        model="nlptown/bert-base-multilingual-uncased-sentiment"
    )

    fin = time.perf_counter()

    tiempo = fin - inicio

    resultados.append({
        "frase": frase,
        "resultado": resultado[0],
        "tiempo_segundos": tiempo
    })

resultados_inference_1= resultados
resultados_inference_1

[{'frase': 'Me encanto el producto, es excelente',
  'resultado': TextClassificationOutputElement(label='5 stars', score=0.841068685054779),
  'tiempo_segundos': 4.535115599996061},
 {'frase': 'No me gusto el producto, la calidad es muy mala',
  'resultado': TextClassificationOutputElement(label='1 star', score=0.7105900049209595),
  'tiempo_segundos': 0.3110490999970352},
 {'frase': 'Es genial, recomendados',
  'resultado': TextClassificationOutputElement(label='5 stars', score=0.7857765555381775),
  'tiempo_segundos': 0.39882159999979194},
 {'frase': 'Es horrible, nunca volveria a comprarlo',
  'resultado': TextClassificationOutputElement(label='1 star', score=0.9611223936080933),
  'tiempo_segundos': 0.30641359998844564},
 {'frase': '.',
  'resultado': TextClassificationOutputElement(label='3 stars', score=0.253933846950531),
  'tiempo_segundos': 0.26207379999686964},
 {'frase': 'Cumple con las 3 B, bueno, bonito, barato',
  'resultado': TextClassificationOutputElement(label='4 star

<h2>Modelo 2 Pipeline vs Inference Client</h2> 

In [75]:
#con pipeline-modelo 2
resultados = []

for frase in frases:
    inicio = time.perf_counter()

    resultado = pipe2(frase)

    fin = time.perf_counter()

    tiempo = fin - inicio

    resultados.append({
        "frase": frase,
        "resultado": resultado[0],
        "tiempo_segundos": tiempo
    })


resultados_pipeline_2 = resultados
resultados_pipeline_2

[{'frase': 'Me encanto el producto, es excelente',
  'resultado': {'label': 'Positive', 'score': 0.7150365114212036},
  'tiempo_segundos': 0.8935147000011057},
 {'frase': 'No me gusto el producto, la calidad es muy mala',
  'resultado': {'label': 'Negative', 'score': 0.8065088391304016},
  'tiempo_segundos': 0.10122599999886006},
 {'frase': 'Es genial, recomendados',
  'resultado': {'label': 'Positive', 'score': 0.826217770576477},
  'tiempo_segundos': 0.1438402999920072},
 {'frase': 'Es horrible, nunca volveria a comprarlo',
  'resultado': {'label': 'Very Negative', 'score': 0.7969605922698975},
  'tiempo_segundos': 0.1557537000044249},
 {'frase': '.',
  'resultado': {'label': 'Neutral', 'score': 0.5881971120834351},
  'tiempo_segundos': 0.02599019999615848},
 {'frase': 'Cumple con las 3 B, bueno, bonito, barato',
  'resultado': {'label': 'Positive', 'score': 0.9343554973602295},
  'tiempo_segundos': 0.04651980000198819},
 {'frase': 'No se, era para un regalo',
  'resultado': {'label'

In [76]:
#con Inference Client -modelo 2
resultados = []

for frase in frases:
    inicio = time.perf_counter()

    resultado = client2.text_classification(
        frase,
        model="tabularisai/multilingual-sentiment-analysis",
    )

    fin = time.perf_counter()

    tiempo = fin - inicio

    resultados.append({
        "frase": frase,
        "resultado": resultado[0],
        "tiempo_segundos": tiempo
    })

resultados_inference_2= resultados
resultados_inference_2

[{'frase': 'Me encanto el producto, es excelente',
  'resultado': TextClassificationOutputElement(label='Very Positive', score=0.6330252885818481),
  'tiempo_segundos': 4.173343100002967},
 {'frase': 'No me gusto el producto, la calidad es muy mala',
  'resultado': TextClassificationOutputElement(label='Very Negative', score=0.49792996048927307),
  'tiempo_segundos': 0.46641700000327546},
 {'frase': 'Es genial, recomendados',
  'resultado': TextClassificationOutputElement(label='Very Positive', score=0.49759441614151),
  'tiempo_segundos': 0.31225370000174735},
 {'frase': 'Es horrible, nunca volveria a comprarlo',
  'resultado': TextClassificationOutputElement(label='Very Negative', score=0.7119160890579224),
  'tiempo_segundos': 0.30324300000211224},
 {'frase': '.',
  'resultado': TextClassificationOutputElement(label='Neutral', score=0.36670804023742676),
  'tiempo_segundos': 0.3080049999989569},
 {'frase': 'Cumple con las 3 B, bueno, bonito, barato',
  'resultado': TextClassificatio

<h2>Tablas comparativas Modelo 1 y Modelo 2 Pipelines</h2>

In [77]:
import pandas as pd

datos_pipeline = [
    # Modelo 1
    {"modelo": "Modelo 1", "frase": "Me encanto el producto, es excelente",
     "label": "5 stars", "score": 0.8410688638687134, "tiempo_segundos": 0.372586900004535},

    {"modelo": "Modelo 1", "frase": "No me gusto el producto, la calidad es muy mala",
     "label": "1 star", "score": 0.7105898261070251, "tiempo_segundos": 0.14169989999209065},

    {"modelo": "Modelo 1", "frase": "Es genial, recomendados",
     "label": "5 stars", "score": 0.7857764959335327, "tiempo_segundos": 0.19785609999962617},

    {"modelo": "Modelo 1", "frase": "Es horrible, nunca volveria a comprarlo",
     "label": "1 star", "score": 0.9611223936080933, "tiempo_segundos": 0.1339250000019092},

    {"modelo": "Modelo 1", "frase": ".",
     "label": "3 stars", "score": 0.2539339065551758, "tiempo_segundos": 0.036762899995665066},

    {"modelo": "Modelo 1", "frase": "Cumple con las 3 B, bueno, bonito, barato",
     "label": "4 stars", "score": 0.4829288721084595, "tiempo_segundos": 0.08941909999703057},

    {"modelo": "Modelo 1", "frase": "No se, era para un regalo",
     "label": "1 star", "score": 0.44101476669311523, "tiempo_segundos": 0.10246840000036173},

    {"modelo": "Modelo 1", "frase": "Lo barato sale caro",
     "label": "2 stars", "score": 0.3705878257751465, "tiempo_segundos": 0.10639260000607464},

    {"modelo": "Modelo 1", "frase": "Ni ACME se atrevio a tanto",
     "label": "2 stars", "score": 0.36325308680534363, "tiempo_segundos": 0.12001379999855999},

    # Modelo 2
    {"modelo": "Modelo 2", "frase": "Me encanto el producto, es excelente",
     "label": "Positive", "score": 0.7150365114212036, "tiempo_segundos": 0.8477190999983577},

    {"modelo": "Modelo 2", "frase": "No me gusto el producto, la calidad es muy mala",
     "label": "Negative", "score": 0.8065088391304016, "tiempo_segundos": 0.07133500000054482},

    {"modelo": "Modelo 2", "frase": "Es genial, recomendados",
     "label": "Positive", "score": 0.826217770576477, "tiempo_segundos": 0.05589540000073612},

    {"modelo": "Modelo 2", "frase": "Es horrible, nunca volveria a comprarlo",
     "label": "Very Negative", "score": 0.7969605922698975, "tiempo_segundos": 0.06582420000631828},

    {"modelo": "Modelo 2", "frase": ".",
     "label": "Neutral", "score": 0.5881971120834351, "tiempo_segundos": 0.020240799989551306},

    {"modelo": "Modelo 2", "frase": "Cumple con las 3 B, bueno, bonito, barato",
     "label": "Positive", "score": 0.9343554973602295, "tiempo_segundos": 0.04820780000591185},

    {"modelo": "Modelo 2", "frase": "No se, era para un regalo",
     "label": "Negative", "score": 0.46413180232048035, "tiempo_segundos": 0.05366449999564793},

    {"modelo": "Modelo 2", "frase": "Lo barato sale caro",
     "label": "Negative", "score": 0.7634744048118591, "tiempo_segundos": 0.03936559999419842},

    {"modelo": "Modelo 2", "frase": "Ni ACME se atrevio a tanto",
     "label": "Neutral", "score": 0.6062771081924438,
     "tiempo_segundos": 0.050909600002341904},
]

df_pipeline = pd.DataFrame(datos_pipeline)

df_pipeline

,modelo,frase,label,score,tiempo_segundos
0,Modelo 1,"Me encanto el producto, es excelente",5 stars,0.841069,0.372587
1,Modelo 1,"No me gusto el producto, la calidad es muy mala",1 star,0.710590,0.141700
2,Modelo 1,"Es genial, recomendados",5 stars,0.785776,0.197856
3,Modelo 1,"Es horrible, nunca volveria a comprarlo",1 star,0.961122,0.133925
4,Modelo 1,.,3 stars,0.253934,0.036763
5,Modelo 1,"Cumple con las 3 B, bueno, bonito, barato",4 stars,0.482929,0.089419
6,Modelo 1,"No se, era para un regalo",1 star,0.441015,0.102468
7,Modelo 1,Lo barato sale caro,2 stars,0.370588,0.106393
8,Modelo 1,Ni ACME se atrevio a tanto,2 stars,0.363253,0.120014
9,Modelo 2,"Me encanto el producto, es excelente",Positive,0.715037,0.847719


<h2>Tablas comparativas Modelo 1 y Modelo 2 Inference Client</h2>

In [78]:
datos_inference = [
    # Modelo 1
    {"modelo": "Modelo 1", "metodo": "InferenceClient",
     "frase": "Me encanto el producto, es excelente",
     "label": "5 stars", "score": 0.841068685054779,
     "tiempo_segundos": 0.3987256999971578},

    {"modelo": "Modelo 1", "metodo": "InferenceClient",
     "frase": "No me gusto el producto, la calidad es muy mala",
     "label": "1 star", "score": 0.7105900049209595,
     "tiempo_segundos": 0.26121739999507554},

    {"modelo": "Modelo 1", "metodo": "InferenceClient",
     "frase": "Es genial, recomendados",
     "label": "5 stars", "score": 0.7857765555381775,
     "tiempo_segundos": 0.2760982000036165},

    {"modelo": "Modelo 1", "metodo": "InferenceClient",
     "frase": "Es horrible, nunca volveria a comprarlo",
     "label": "1 star", "score": 0.9611223936080933,
     "tiempo_segundos": 0.2722458999924129},

    {"modelo": "Modelo 1", "metodo": "InferenceClient",
     "frase": ".",
     "label": "3 stars", "score": 0.253933846950531,
     "tiempo_segundos": 0.23391429999901447},

    {"modelo": "Modelo 1", "metodo": "InferenceClient",
     "frase": "Cumple con las 3 B, bueno, bonito, barato",
     "label": "4 stars", "score": 0.4829288721084595,
     "tiempo_segundos": 0.241403399995761},

    {"modelo": "Modelo 1", "metodo": "InferenceClient",
     "frase": "No se, era para un regalo",
     "label": "1 star", "score": 0.44101482629776,
     "tiempo_segundos": 0.24169260000053328},

    {"modelo": "Modelo 1", "metodo": "InferenceClient",
     "frase": "Lo barato sale caro",
     "label": "2 stars", "score": 0.37058785557746887,
     "tiempo_segundos": 0.27957559999777004},

    {"modelo": "Modelo 1", "metodo": "InferenceClient",
     "frase": "Ni ACME se atrevio a tanto",
     "label": "2 stars", "score": 0.3632528483867645,
     "tiempo_segundos": 0.24194409999472555},

    # Modelo 2
    {"modelo": "Modelo 2", "metodo": "InferenceClient",
     "frase": "Me encanto el producto, es excelente",
     "label": "Very Positive", "score": 0.6330252885818481,
     "tiempo_segundos": 7.723080099996878},

    {"modelo": "Modelo 2", "metodo": "InferenceClient",
     "frase": "No me gusto el producto, la calidad es muy mala",
     "label": "Very Negative", "score": 0.49792996048927307,
     "tiempo_segundos": 2.760539199996856},

    {"modelo": "Modelo 2", "metodo": "InferenceClient",
     "frase": "Es genial, recomendados",
     "label": "Very Positive", "score": 0.49759441614151,
     "tiempo_segundos": 2.5153254000033485},

    {"modelo": "Modelo 2", "metodo": "InferenceClient",
     "frase": "Es horrible, nunca volveria a comprarlo",
     "label": "Very Negative", "score": 0.7119160890579224,
     "tiempo_segundos": 2.172371100008604},

    {"modelo": "Modelo 2", "metodo": "InferenceClient",
     "frase": ".",
     "label": "Neutral", "score": 0.36670804023742676,
     "tiempo_segundos": 2.3785150999901816},

    {"modelo": "Modelo 2", "metodo": "InferenceClient",
     "frase": "Cumple con las 3 B, bueno, bonito, barato",
     "label": "Positive", "score": 0.6444312334060669,
     "tiempo_segundos": 1.9563582999981008},

    {"modelo": "Modelo 2", "metodo": "InferenceClient",
     "frase": "No se, era para un regalo",
     "label": "Neutral", "score": 0.3554631471633911,
     "tiempo_segundos": 1.0071945999952732},

    {"modelo": "Modelo 2", "metodo": "InferenceClient",
     "frase": "Lo barato sale caro",
     "label": "Neutral", "score": 0.36566615104675293,
     "tiempo_segundos": 0.49845270000514574},

    {"modelo": "Modelo 2", "metodo": "InferenceClient",
     "frase": "Ni ACME se atrevio a tanto",
     "label": "Negative", "score": 0.40250563621520996,
     "tiempo_segundos": 2.0353084999951534},
]

df_inference = pd.DataFrame(datos_inference)

df_inference

,modelo,metodo,frase,label,score,tiempo_segundos
0,Modelo 1,InferenceClient,"Me encanto el producto, es excelente",5 stars,0.841069,0.398726
1,Modelo 1,InferenceClient,"No me gusto el producto, la calidad es muy mala",1 star,0.710590,0.261217
2,Modelo 1,InferenceClient,"Es genial, recomendados",5 stars,0.785777,0.276098
3,Modelo 1,InferenceClient,"Es horrible, nunca volveria a comprarlo",1 star,0.961122,0.272246
4,Modelo 1,InferenceClient,.,3 stars,0.253934,0.233914
5,Modelo 1,InferenceClient,"Cumple con las 3 B, bueno, bonito, barato",4 stars,0.482929,0.241403
6,Modelo 1,InferenceClient,"No se, era para un regalo",1 star,0.441015,0.241693
7,Modelo 1,InferenceClient,Lo barato sale caro,2 stars,0.370588,0.279576
8,Modelo 1,InferenceClient,Ni ACME se atrevio a tanto,2 stars,0.363253,0.241944
9,Modelo 2,InferenceClient,"Me encanto el producto, es excelente",Very Positive,0.633025,7.723080


<u>  Tipos de datos:</u>

In [85]:
#pipelines modelos 1 y 2
print(type(resultado_pipeline_1))
print(type(resultado_pipeline_1[0]))
print(resultado_pipeline_1[0])

print(resultado_pipeline_1[0]["label"])
print(resultado_pipeline_1[0]["score"])


print(type(resultado_pipeline_2))
print(type(resultado_pipeline_2[0]))
print(resultado_pipeline_2[0])

print(resultado_pipeline_2[0]["label"])
print(resultado_pipeline_2[0]["score"])

<class 'list'>
<class 'dict'>
{'label': '5 stars', 'score': 0.8544487357139587}
5 stars
0.8544487357139587
<class 'list'>
<class 'dict'>
{'label': 'Positive', 'score': 0.8007412552833557}
Positive
0.8007412552833557


In [90]:
#inference client
print(type(resultados_inference_1))
print(type(resultados_inference_1[0]))
print(resultados_inference_1[0])

print(type(resultados_inference_1[0]["resultado"]))
print(resultados_inference_1[0]["resultado"])

print(resultados_inference_1[0]["resultado"].label)
print(resultados_inference_1[0]["resultado"].score)

<class 'list'>
<class 'dict'>
{'frase': 'Me encanto el producto, es excelente', 'resultado': TextClassificationOutputElement(label='5 stars', score=0.841068685054779), 'tiempo_segundos': 4.535115599996061}
<class 'huggingface_hub.inference._generated.types.text_classification.TextClassificationOutputElement'>
TextClassificationOutputElement(label='5 stars', score=0.841068685054779)
5 stars
0.841068685054779


<h2>Sentimiento Esperado</h2>

In [109]:
sentimiento_esperado = {
    "Me encanto el producto, es excelente": "Muy positivo",
    "No me gusto el producto, la calidad es muy mala": "Muy negativo",
    "Es genial, recomendados": "Positivo",
    "Es horrible, nunca volveria a comprarlo": "Muy negativo",
    ".": "Neutral",
    "Cumple con las 3 B, bueno, bonito, barato": "Positivo",
    "No se, era para un regalo": "Neutral",
    "Lo barato sale caro": "Negativo",
    "Ni ACME se atrevio a tanto": "Negativo"
}

# Pasar las etiquetas del Modelo 2 a español
mapa_modelo_2 = {
    "Very Negative": "Muy negativo",
    "Negative": "Negativo",
    "Neutral": "Neutral",
    "Positive": "Positivo",
    "Very Positive": "Muy positivo"
}

# Convertir las estrellas del Modelo 1 a sentimiento
mapa_modelo_1 = {
    "1 star": "Muy negativo",
    "2 stars": "Negativo",
    "3 stars": "Neutral",
    "4 stars": "Positivo",
    "5 stars": "Muy positivo"
}

mapa_polaridad = {
    "Muy positivo": "Positivo",
    "Positivo": "Positivo",
    "Neutral": "Neutral",
    "Negativo": "Negativo",
    "Muy negativo": "Negativo"
}

In [111]:
#agregamos los sentimientos esperados y el mapeo de estrellas y a español
def evaluar_pipeline(fila):
    esperado = sentimiento_esperado[fila["frase"]]

    if fila["modelo"] == "Modelo 1":
        prediccion = mapa_modelo_1[fila["label"]]
    else:
        prediccion = mapa_modelo_2[fila["label"]]

    esperado_polaridad = mapa_polaridad[esperado]
    prediccion_polaridad = mapa_polaridad[prediccion]

    return "Sí" if prediccion_polaridad == esperado_polaridad else "No"


df_pipeline["correcto"] = df_pipeline.apply(evaluar_pipeline, axis=1)

df_pipeline

,modelo,frase,label,score,tiempo_segundos,correcto
0,Modelo 1,"Me encanto el producto, es excelente",5 stars,0.841069,0.372587,Sí
1,Modelo 1,"No me gusto el producto, la calidad es muy mala",1 star,0.710590,0.141700,Sí
2,Modelo 1,"Es genial, recomendados",5 stars,0.785776,0.197856,Sí
3,Modelo 1,"Es horrible, nunca volveria a comprarlo",1 star,0.961122,0.133925,Sí
4,Modelo 1,.,3 stars,0.253934,0.036763,Sí
5,Modelo 1,"Cumple con las 3 B, bueno, bonito, barato",4 stars,0.482929,0.089419,Sí
6,Modelo 1,"No se, era para un regalo",1 star,0.441015,0.102468,No
7,Modelo 1,Lo barato sale caro,2 stars,0.370588,0.106393,Sí
8,Modelo 1,Ni ACME se atrevio a tanto,2 stars,0.363253,0.120014,Sí
9,Modelo 2,"Me encanto el producto, es excelente",Positive,0.715037,0.847719,Sí


In [112]:
def evaluar_inference(fila):
    esperado = sentimiento_esperado[fila["frase"]]

    if fila["modelo"] == "Modelo 1":
        prediccion = mapa_modelo_1[fila["label"]]
    else:
        prediccion = mapa_modelo_2[fila["label"]]

    esperado_polaridad = mapa_polaridad[esperado]
    prediccion_polaridad = mapa_polaridad[prediccion]

    return "Sí" if prediccion_polaridad == esperado_polaridad else "No"


df_inference["correcto"] = df_inference.apply(evaluar_inference, axis=1)

df_inference

,modelo,metodo,frase,label,score,tiempo_segundos,correcto
0,Modelo 1,InferenceClient,"Me encanto el producto, es excelente",5 stars,0.841069,0.398726,Sí
1,Modelo 1,InferenceClient,"No me gusto el producto, la calidad es muy mala",1 star,0.710590,0.261217,Sí
2,Modelo 1,InferenceClient,"Es genial, recomendados",5 stars,0.785777,0.276098,Sí
3,Modelo 1,InferenceClient,"Es horrible, nunca volveria a comprarlo",1 star,0.961122,0.272246,Sí
4,Modelo 1,InferenceClient,.,3 stars,0.253934,0.233914,Sí
5,Modelo 1,InferenceClient,"Cumple con las 3 B, bueno, bonito, barato",4 stars,0.482929,0.241403,Sí
6,Modelo 1,InferenceClient,"No se, era para un regalo",1 star,0.441015,0.241693,No
7,Modelo 1,InferenceClient,Lo barato sale caro,2 stars,0.370588,0.279576,Sí
8,Modelo 1,InferenceClient,Ni ACME se atrevio a tanto,2 stars,0.363253,0.241944,Sí
9,Modelo 2,InferenceClient,"Me encanto el producto, es excelente",Very Positive,0.633025,7.723080,Sí


<h2>Que Modelo usar?</h2>

<h3>Comparacion de porcentajes de aciertos</h3>

In [118]:
resumen_pipeline = (
    df_pipeline[df_pipeline["correcto"] != "N/A"]
    .groupby("modelo")["correcto"]
    .apply(lambda x: (x == "Sí").mean() * 100)
    .reset_index(name="porcentaje_aciertos")
)

resumen_pipeline

,modelo,porcentaje_aciertos
0,Modelo 1,88.888889
1,Modelo 2,77.777778


Pipeline de Modelo 1 tiene mas aciertos con un porcentaje de 88,89%

In [114]:
resumen_inference = (
    df_inference[df_inference["correcto"] != "N/A"]
    .groupby("modelo")["correcto"]
    .apply(lambda x: (x == "Sí").mean() * 100)
    .reset_index(name="porcentaje_aciertos")
)

resumen_inference

,modelo,porcentaje_aciertos
0,Modelo 1,88.888889
1,Modelo 2,88.888889


Inference son iguales en ambos modelos

<h3> Resumen de tiempos promedio </h3>

In [116]:
tiempo_pipeline = (
    df_pipeline
    .groupby("modelo")["tiempo_segundos"]
    .mean()
    .reset_index(name="tiempo_promedio")
)

tiempo_pipeline

,modelo,tiempo_promedio
0,Modelo 1,0.144569
1,Modelo 2,0.139240


El modelo 2 de pipeline presenta menor tiempo de ejecucion promedio

In [117]:
tiempo_inference = (
    df_inference
    .groupby("modelo")["tiempo_segundos"]
    .mean()
    .reset_index(name="tiempo_promedio")
)

tiempo_inference

,modelo,tiempo_promedio
0,Modelo 1,0.271869
1,Modelo 2,2.560794


Mientras que en el inference el modelo 1 es mas rapido
